# Navier–Stokes PINNsFormer 梯度冲突诊断

目的：**先不引入 ConFIG**，对已经训练过的 PINNsFormer checkpoint 计算 `L_data` 与 `L_physics` 的参数梯度范数、cosine 和冲突情况，并额外给出四项 `u_data / v_data / f_u / f_v` 的 cosine matrix。

与上传 notebook 保持的数据、网络、随机种子、`N_TRAIN=800`、5-step pseudo sequence、Navier–Stokes residual 不变。此 notebook 不重新训练。

> 注意：上传 notebook / 上游 Navier–Stokes notebook 的测试 cell 与训练 cell 在 stream-function 的 x/y 导数上不一致。本文件只做训练损失梯度诊断，始终按训练定义 `u=psi_y, v=-psi_x`。


In [1]:
from pathlib import Path
import os
import sys
import time
import random

import numpy as np
import scipy.io
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm

# Paths are configurable; defaults match the Linux server used for the previous run.
PINNSFORMER_ROOT = Path(os.environ.get("PINNSFORMER_ROOT", "/home/simplexity/cyt/pinnsformer-main"))
CONFIG_ROOT = Path(os.environ.get("CONFIG_ROOT", "/home/simplexity/cyt/ConFIG-main"))
NAVIER_DIR = PINNSFORMER_ROOT / "demo" / "navier_stokes"
DATA_PATH = NAVIER_DIR / "cylinder_nektar_wake.mat"

sys.path.insert(0, str(PINNSFORMER_ROOT))
sys.path.insert(0, str(CONFIG_ROOT))

from util import get_clones, get_n_params
from gradient_diagnostics import gradient_vector, summarize_gradients, ConflictTracker

assert DATA_PATH.exists(), f"Data not found: {DATA_PATH}"
print("PINNsFormer root:", PINNSFORMER_ROOT)
print("ConFIG root     :", CONFIG_ROOT)
print("Data            :", DATA_PATH)


PINNsFormer root: /home/simplexity/cyt/pinnsformer-main
ConFIG root     : /home/simplexity/cyt/ConFIG-main
Data            : /home/simplexity/cyt/pinnsformer-main/demo/navier_stokes/cylinder_nektar_wake.mat


In [2]:
from util import make_time_sequence
SEED = 0
DEVICE = "cuda:0"

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

assert torch.cuda.is_available(), "CUDA is required for this experiment."
device = torch.device(DEVICE)
print("Device:", torch.cuda.get_device_name(device))


Device: NVIDIA GeForce GTX 1080 Ti


In [3]:
data = scipy.io.loadmat(DATA_PATH)

U_star = data["U_star"]  # N x 2 x T
P_star = data["p_star"]  # N x T
t_star_raw = data["t"]   # T x 1
X_star = data["X_star"]  # N x 2

N = X_star.shape[0]
T = t_star_raw.shape[0]

XX = np.tile(X_star[:, 0:1], (1, T))
YY = np.tile(X_star[:, 1:2], (1, T))
TT = np.tile(t_star_raw, (1, N)).T
UU = U_star[:, 0, :]
VV = U_star[:, 1, :]
PP = P_star

x = XX.flatten()[:, None]
y = YY.flatten()[:, None]
t = TT.flatten()[:, None]
u = UU.flatten()[:, None]
v = VV.flatten()[:, None]
p = PP.flatten()[:, None]

# Keep the uploaded run settings unchanged.
N_TRAIN = 800
NUM_STEP = 5
TIME_STEP = 1e-2

idx = np.random.choice(N * T, N_TRAIN, replace=False)
x_train_np = x[idx, :]
y_train_np = y[idx, :]
t_train_np = t[idx, :]
u_train_np = u[idx, :]
v_train_np = v[idx, :]

x_train_np = np.expand_dims(np.tile(x_train_np[:], NUM_STEP), -1)
y_train_np = np.expand_dims(np.tile(y_train_np[:], NUM_STEP), -1)
t_train_np = make_time_sequence(t_train_np, num_step=NUM_STEP, step=TIME_STEP)

x_train = torch.tensor(x_train_np, dtype=torch.float32, requires_grad=True, device=device)
y_train = torch.tensor(y_train_np, dtype=torch.float32, requires_grad=True, device=device)
t_train = torch.tensor(t_train_np, dtype=torch.float32, requires_grad=True, device=device)
u_train = torch.tensor(u_train_np, dtype=torch.float32, device=device)
v_train = torch.tensor(v_train_np, dtype=torch.float32, device=device)

print(f"Training points: {N_TRAIN} (total available: {N*T})")
print("Sequence shape :", x_train.shape)


Training points: 800 (total available: 1000000)
Sequence shape : torch.Size([800, 5, 1])


In [4]:
class WaveAct(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 = nn.Parameter(torch.ones(1), requires_grad=True)
        self.w2 = nn.Parameter(torch.ones(1), requires_grad=True)

    def forward(self, x):
        return self.w1 * torch.sin(x) + self.w2 * torch.cos(x)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=256):
        super().__init__()
        self.linear = nn.Sequential(
            nn.Linear(d_model, d_ff),
            WaveAct(),
            nn.Linear(d_ff, d_ff),
            WaveAct(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.linear(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)
        self.ff = FeedForward(d_model)
        self.act1 = WaveAct()
        self.act2 = WaveAct()

    def forward(self, x):
        x2 = self.act1(x)
        x = x + self.attn(x2, x2, x2)[0]
        x2 = self.act2(x)
        x = x + self.ff(x2)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=heads, batch_first=True)
        self.ff = FeedForward(d_model)
        self.act1 = WaveAct()
        self.act2 = WaveAct()

    def forward(self, x, e_outputs):
        x2 = self.act1(x)
        x = x + self.attn(x2, e_outputs, e_outputs)[0]
        x2 = self.act2(x)
        x = x + self.ff(x2)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model, n_layers, heads):
        super().__init__()
        self.n_layers = n_layers
        self.layers = get_clones(EncoderLayer(d_model, heads), n_layers)
        self.act = WaveAct()

    def forward(self, x):
        for i in range(self.n_layers):
            x = self.layers[i](x)
        return self.act(x)


class Decoder(nn.Module):
    def __init__(self, d_model, n_layers, heads):
        super().__init__()
        self.n_layers = n_layers
        self.layers = get_clones(DecoderLayer(d_model, heads), n_layers)
        self.act = WaveAct()

    def forward(self, x, e_outputs):
        for i in range(self.n_layers):
            x = self.layers[i](x, e_outputs)
        return self.act(x)


class PINNsformer(nn.Module):
    def __init__(self, d_out, d_model, d_hidden, N, heads):
        super().__init__()
        self.linear_emb = nn.Linear(3, d_model)
        self.encoder = Encoder(d_model, N, heads)
        self.decoder = Decoder(d_model, N, heads)
        self.linear_out = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            WaveAct(),
            nn.Linear(d_hidden, d_hidden),
            WaveAct(),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x, y, t):
        src = torch.cat((x, y, t), dim=-1)
        src = self.linear_emb(src)
        e_outputs = self.encoder(src)
        d_output = self.decoder(src, e_outputs)
        return self.linear_out(d_output)


def init_weights(m):
    if isinstance(m, nn.Linear):
        # The uploaded notebook used the deprecated alias xavier_uniform.
        # xavier_uniform_ is the current in-place equivalent.
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)


In [5]:
def compute_ns_losses(model, x_in, y_in, t_in, u_obs, v_obs):
    psi_and_p = model(x_in, y_in, t_in)
    psi = psi_and_p[:, :, 0:1]
    pressure = psi_and_p[:, :, 1:2]

    # Keep the training stream-function convention from the uploaded notebook:
    # u = dpsi/dy, v = -dpsi/dx.
    u_pred = torch.autograd.grad(
        psi, y_in, grad_outputs=torch.ones_like(psi), retain_graph=True, create_graph=True
    )[0]
    v_pred = -torch.autograd.grad(
        psi, x_in, grad_outputs=torch.ones_like(psi), retain_graph=True, create_graph=True
    )[0]

    u_t = torch.autograd.grad(u_pred, t_in, grad_outputs=torch.ones_like(u_pred), retain_graph=True, create_graph=True)[0]
    u_x = torch.autograd.grad(u_pred, x_in, grad_outputs=torch.ones_like(u_pred), retain_graph=True, create_graph=True)[0]
    u_y = torch.autograd.grad(u_pred, y_in, grad_outputs=torch.ones_like(u_pred), retain_graph=True, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_in, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y_in, grad_outputs=torch.ones_like(u_y), retain_graph=True, create_graph=True)[0]

    v_t = torch.autograd.grad(v_pred, t_in, grad_outputs=torch.ones_like(v_pred), retain_graph=True, create_graph=True)[0]
    v_x = torch.autograd.grad(v_pred, x_in, grad_outputs=torch.ones_like(v_pred), retain_graph=True, create_graph=True)[0]
    v_y = torch.autograd.grad(v_pred, y_in, grad_outputs=torch.ones_like(v_pred), retain_graph=True, create_graph=True)[0]
    v_xx = torch.autograd.grad(v_x, x_in, grad_outputs=torch.ones_like(v_x), retain_graph=True, create_graph=True)[0]
    v_yy = torch.autograd.grad(v_y, y_in, grad_outputs=torch.ones_like(v_y), retain_graph=True, create_graph=True)[0]

    p_x = torch.autograd.grad(pressure, x_in, grad_outputs=torch.ones_like(pressure), retain_graph=True, create_graph=True)[0]
    p_y = torch.autograd.grad(pressure, y_in, grad_outputs=torch.ones_like(pressure), retain_graph=True, create_graph=True)[0]

    f_u = u_t + (u_pred * u_x + v_pred * u_y) + p_x - 0.01 * (u_xx + u_yy)
    f_v = v_t + (u_pred * v_x + v_pred * v_y) + p_y - 0.01 * (v_xx + v_yy)

    # Data terms use the first element of each 5-step pseudo sequence, exactly as uploaded.
    loss_u_data = torch.mean((u_pred[:, 0] - u_obs) ** 2)
    loss_v_data = torch.mean((v_pred[:, 0] - v_obs) ** 2)
    loss_fu = torch.mean(f_u ** 2)
    loss_fv = torch.mean(f_v ** 2)

    loss_data = loss_u_data + loss_v_data
    loss_physics = loss_fu + loss_fv
    loss_total = loss_data + loss_physics

    return {
        "total": loss_total,
        "data": loss_data,
        "physics": loss_physics,
        "u_data": loss_u_data,
        "v_data": loss_v_data,
        "f_u": loss_fu,
        "f_v": loss_fv,
    }


In [6]:
CHECKPOINT_PATH = NAVIER_DIR / "ns_pinnsformer.pt"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

model = PINNsformer(d_out=2, d_hidden=512, d_model=32, N=1, heads=2).to(device)
model.apply(init_weights)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

print("Loaded:", CHECKPOINT_PATH)
print("Parameters:", get_n_params(model))


Loaded: /home/simplexity/cyt/pinnsformer-main/demo/navier_stokes/ns_pinnsformer.pt
Parameters: 454106


In [7]:
losses = compute_ns_losses(model, x_train, y_train, t_train, u_train, v_train)

# 2-loss diagnosis: data vs physics
g_data = gradient_vector(losses["data"], model, retain_graph=True)
g_physics = gradient_vector(losses["physics"], model, retain_graph=True)
two = summarize_gradients({"data": g_data, "physics": g_physics})

print("2-loss values:")
print("  L_data   =", float(losses["data"].detach()))
print("  L_physics=", float(losses["physics"].detach()))
print("2-loss gradient norms:", two["norms"])
print("2-loss cosine matrix:\n", two["cosine_matrix"])
print("Conflict:", bool(two["cosine_matrix"][0, 1] < 0))


2-loss values:
  L_data   = 1.4945510429242859e-06
  L_physics= 1.1262984571658308e-06
2-loss gradient norms: {'data': 0.00032983109122142196, 'physics': 0.00033898564288392663}
2-loss cosine matrix:
 [[ 1.         -0.92318034]
 [-0.92318034  1.        ]]
Conflict: True


In [8]:
# 4-loss diagnosis using the same forward graph.
# Recompute once so graph lifetime is explicit and easy to reason about.
losses = compute_ns_losses(model, x_train, y_train, t_train, u_train, v_train)
names = ["u_data", "v_data", "f_u", "f_v"]
grads = {}
for k, name in enumerate(names):
    grads[name] = gradient_vector(losses[name], model, retain_graph=(k < len(names) - 1))

four = summarize_gradients(grads)
print("4-loss gradient norms:")
for k, v_ in four["norms"].items():
    print(f"  {k:>8s}: {v_:.6e}")
print("Names:", four["names"])
print("Cosine matrix:\n", four["cosine_matrix"])
print("Pair conflict rate:", four["pair_conflict_rate"])


RuntimeError: CUDA out of memory. Tried to allocate 20.00 MiB (GPU 0; 10.91 GiB total capacity; 9.39 GiB already allocated; 47.44 MiB free; 9.42 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF